# Valoración de Servicios Ecosistémicos — Parque Eólico Paxareiras II

**Expediente:** IN661A 2011/8 (NT)  
**Localización:** Dumbría, A Coruña, Galicia  
**Potencia instalada:** 26,40 MW (3×5,0 MW + 2×5,7 MW — modelo Nordex N163)  
**Autor:** Fernando García Abal — Ingeniero Agrónomo / Ingeniero de Montes  
**Marco metodológico:** Benefit Transfer + VAN 25 años (tasa 3%)  
**Fuentes de datos:**
- Coordenadas: Acordo IP Xunta de Galicia, octubre 2025 (CVE: bWmKo5yl53t5)
- Usos del suelo: CORINE Land Cover 2018 — IGN/Copernicus
- Espacios protegidos: Red Natura 2000 — shapefile
- Valores monetarios: síntesis del analista a partir de Costanza et al. (2014), de Groot et al. (2012) y Quintas-Soriano et al. (2016) — ver sección 14 (Limitaciones)

**Fecha:** Junio 2026  

## Finalidad y alcance

**Objetivo de este trabajo.** Demostrar la capacidad de **integrar GIS + Python + valoración económica de servicios ecosistémicos** en un flujo de trabajo reproducible, aplicado a un expediente real. Es una **pieza de portfolio técnico**, no un informe pericial.

**Naturaleza del resultado.** Las cifras económicas son una **aproximación de primer orden** — una contabilidad de capital natural de línea base mediante transferencia de valores unitarios. Sirven para ilustrar el método y dar un orden de magnitud; **no** constituyen una valoración de impacto apta para fundamentar una decisión de EIA. Los supuestos y limitaciones que acotan esta aproximación se detallan en la sección 14.

**Línea base ≠ impacto neto.** Las secciones 8–9 cuantifican el valor de contexto territorial del área de influencia; la sección 10 añade el valor contenido en la poligonal como proxy de superficie. El impacto neto (huella física, fragmentación, colisión de avifauna, paisaje, y el contrapeso del beneficio climático de la generación renovable) excede el alcance de esta pieza y se aborda en piezas posteriores.

---

## Referencia regulatoria
Ley 2/2024 de medidas para la simplificación y agilización administrativa de Galicia,  
que incorpora la valoración de servicios ecosistémicos en los procedimientos de EIA.

---

## Estructura
1. Configuración del entorno  
2. Definición del área de estudio (coordenadas oficiales)  
3. Carga del CORINE Land Cover 2018  
4. Recorte al área de influencia  
5. Cálculo de superficies por clase de uso  
6. Red Natura 2000 — contexto espacial  
7. Asignación de servicios ecosistémicos  
8. Valoración monetaria por benefit transfer (línea base)  
9. VAN a 25 años (tasa 3%) + análisis de sensibilidad  
10. Valor contenido en la poligonal (proxy de superficie afectada)  
11. Tabla ejecutiva de resultados  
12. Exportación a Excel  
13. Visualizaciones de publicación  
14. Limitaciones metodológicas y referencias  


---
## 1. Configuración del entorno

In [ ]:
# Instalar dependencias si es necesario (descomentar)
# !pip install geopandas shapely matplotlib contextily openpyxl

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from shapely.geometry import Point, Polygon, LineString
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

try:
    import contextily as ctx
    CONTEXTILY = True
except ImportError:
    CONTEXTILY = False
    print("Nota: contextily no disponible — mapas sin fondo cartográfico.")

print(f"GeoPandas {gpd.__version__} — entorno listo.")

# Inicialización de variables (evita NameError si faltan los shapefiles)
clc_raw = None
clc = None
natura = None

---
## 2. Definición del área de estudio

Coordenadas extraídas directamente del **Acordo de Información Pública**  
firmado el 08/10/2025 por la Directora Territorial de A Coruña (CVE: bWmKo5yl53t5).  
Sistema de referencia: **UTM Fuso 29 ETRS89 (EPSG:25829)**.

In [ ]:
CRS = "EPSG:25829"  # UTM zona 29N ETRS89 — estándar Galicia

# ── POLIGONAL OFICIAL DEL PARQUE (10 vértices) ──────────────────────────────
poligonal_coords = [
    (485855.65, 4759665.77),  # A
    (487531.22, 4760695.47),  # B
    (488194.66, 4760695.55),  # C
    (490320.19, 4760056.43),  # D
    (490330.21, 4759392.56),  # E
    (490953.37, 4759392.56),  # F
    (491788.07, 4758733.71),  # G
    (491378.77, 4758488.13),  # H
    (486637.52, 4758488.22),  # I
    (485855.99, 4759056.33),  # J
    (485855.65, 4759665.77),  # cierre
]
poligonal = gpd.GeoDataFrame(
    {"nombre": ["Paxareiras II"]},
    geometry=[Polygon(poligonal_coords)],
    crs=CRS
)

# ── AEROGENERADORES ─────────────────────────────────────────────────────────
aerox_data = {
    "id":   ["PAX-01","PAX-02","PAX-03","PAX-04","PAX-05"],
    "mw":   [5.0, 5.0, 5.0, 5.7, 5.7],
    "x":    [487684.35, 488184.36, 488754.36, 489150.87, 489861.38],
    "y":    [4760425.48, 4760217.48, 4760304.48, 4760013.76, 4759587.48],
}
aeroxeradores = gpd.GeoDataFrame(
    aerox_data,
    geometry=gpd.points_from_xy(aerox_data["x"], aerox_data["y"]),
    crs=CRS
)

# ── LÍNEA DE EVACUACIÓN ─────────────────────────────────────────────────────
evac_coords = [
    (491611.47, 4758774.65),
    (493354.51, 4758446.95),
    (493415.39, 4759396.46),
    (495900.54, 4762304.51),
    (495971.92, 4762233.92),
]
linea_evac = gpd.GeoDataFrame(
    {"nombre": ["LAT 66kV evacuación"]},
    geometry=[LineString(evac_coords)],
    crs=CRS
)

# ── ÁREA DE INFLUENCIA (5 km desde centroide de la poligonal) ───────────────
centroide = poligonal.geometry.centroid.iloc[0]
RADIO_M = 5000
area_influencia = gpd.GeoDataFrame(
    {"nombre": ["Área influencia 5km"]},
    geometry=[centroide.buffer(RADIO_M)],
    crs=CRS
)

sup_poligonal = poligonal.geometry.area.sum() / 10_000
sup_influencia = area_influencia.geometry.area.sum() / 10_000

print(f"Centroide del parque:       X={centroide.x:.0f}  Y={centroide.y:.0f}")
print(f"Superficie poligonal:       {sup_poligonal:.1f} ha")
print(f"Área de influencia (5 km):  {sup_influencia:.1f} ha")
print(f"Aerogeneradores:            {len(aeroxeradores)} unidades — {aerox_data['mw']} MW")
print(f"Longitud línea evacuación:  {linea_evac.geometry.length.sum()/1000:.2f} km")

---
## 3. Carga del CORINE Land Cover 2018

**Archivo esperado:** `data/CLC2018_ZONA_CEE_DUMBRIA.shp` (carpeta `data/` del repositorio).  
**Fuente:** IGN — Centro de Descargas: https://centrodedescargas.cnig.es (serie 02113)


In [ ]:
# ── AJUSTA AQUÍ la ruta de tu shapefile CORINE ──────────────────────────────
RUTA_CORINE = Path("data/CLC2018_ZONA_CEE_DUMBRIA.shp")   # ruta relativa del repo
# ────────────────────────────────────────────────────────────────────────────

if not RUTA_CORINE.exists():
    # Buscar automáticamente cualquier .shp que contenga 'clc' o 'corine' en . y data/
    patrones = ["*[Cc][Ll][Cc]*.shp", "*[Cc][Oo][Rr][Ii][Nn][Ee]*.shp"]
    candidatos = []
    for base in (Path("."), Path("data")):
        for p in patrones:
            candidatos += list(base.glob(p))
    if candidatos:
        RUTA_CORINE = candidatos[0]
        print(f"Encontrado automáticamente: {RUTA_CORINE}")
    else:
        print("⚠️  Shapefile CORINE no encontrado.")
        print(f"   Busca en esta carpeta: {Path('.').resolve()}")
        print("   Ajusta la variable RUTA_CORINE con la ruta exacta.")

if RUTA_CORINE.exists():
    print(f"Cargando {RUTA_CORINE.name} ...")

    # Recorte por bounding box antes de cargar todo España (más rápido)
    bbox_utm = area_influencia.total_bounds  # [xmin, ymin, xmax, ymax]

    clc_raw = gpd.read_file(RUTA_CORINE, bbox=tuple(bbox_utm))

    if clc_raw.crs != area_influencia.crs:
        clc_raw = clc_raw.to_crs(CRS)

    print(f"  Registros en bbox:  {len(clc_raw)}")
    print(f"  CRS reproyectado:   {CRS}")
    print(f"  Columnas:           {list(clc_raw.columns)}")
    clc_raw.head(3)

---
## 4. Recorte al área de influencia

In [ ]:
if clc_raw is not None and len(clc_raw) > 0:
    clc = gpd.clip(clc_raw, area_influencia)
    clc = clc.copy()
    clc["sup_ha"] = clc.geometry.area / 10_000

    # Detectar columna con código CLC (puede llamarse CODE_18, CLC_CODE, etc.)
    col_codigo = next((c for c in clc.columns
                       if "CODE" in c.upper() or "CLC" in c.upper()), clc.columns[0])
    col_label  = next((c for c in clc.columns
                       if "LABEL" in c.upper() or "DESC" in c.upper()), None)

    print(f"Columna código CLC detectada: '{col_codigo}'")
    print(f"Columna descripción detectada: '{col_label}'")
    print(f"Registros tras recorte: {len(clc)}")
    print(f"Clases CLC presentes:   {clc[col_codigo].nunique()}")
    print(f"Superficie total:       {clc['sup_ha'].sum():.1f} ha")

---
## 5. Superficies por clase de uso del suelo

In [ ]:
if clc is not None:
    grp_cols = [col_codigo] + ([col_label] if col_label else [])
    tabla_sup = (
        clc.groupby(grp_cols)["sup_ha"]
        .sum()
        .reset_index()
        .sort_values("sup_ha", ascending=False)
        .reset_index(drop=True)
    )
    sup_total = tabla_sup["sup_ha"].sum()
    tabla_sup["%_total"] = (tabla_sup["sup_ha"] / sup_total * 100).round(2)
    tabla_sup["sup_ha"]  = tabla_sup["sup_ha"].round(2)

    print(f"Superficie total analizada: {sup_total:.1f} ha")
    print("="*65)
    display(tabla_sup)

---
## 6. Red Natura 2000 — contexto espacial

Carga la capa de Red Natura 2000 y calcula la distancia mínima  
entre el parque y el espacio protegido más próximo.

In [ ]:
# ── AJUSTA AQUÍ el nombre de tu shapefile Red Natura ────────────────────────
RUTA_NATURA = Path("data/ZEC_CEE-DUMBRIA.shp")   # ruta relativa del repo
# ────────────────────────────────────────────────────────────────────────────

if not RUTA_NATURA.exists():
    candidatos_n = list(Path(".").glob("*[Nn]atura*.shp")) + \
                   list(Path(".").glob("*[Nn]2000*.shp")) + \
                   list(Path(".").glob("*[Zz][Ee][Pp][Aa]*.shp")) + \
                   list(Path(".").glob("*[Ll][Ii][Cc]*.shp"))
    if candidatos_n:
        RUTA_NATURA = candidatos_n[0]
        print(f"Encontrado automáticamente: {RUTA_NATURA}")
    else:
        print("⚠️  Shapefile Red Natura no encontrado. Ajusta RUTA_NATURA.")
        natura = None

if RUTA_NATURA.exists():
    print(f"Cargando {RUTA_NATURA.name} ...")
    natura_raw = gpd.read_file(RUTA_NATURA, bbox=tuple(area_influencia.buffer(10000).total_bounds))

    if natura_raw.crs != area_influencia.crs:
        natura_raw = natura_raw.to_crs(CRS)

    # Recortar a área de influencia extendida (10 km para contexto)
    natura = gpd.clip(natura_raw, area_influencia.buffer(10000))

    print(f"  Espacios Natura en radio 10km: {len(natura)}")
    if len(natura) > 0:
        print(f"  Columnas: {list(natura.columns)}")

        # Distancia mínima entre poligonal del parque y Red Natura
        dist_min = natura.geometry.distance(poligonal.geometry.iloc[0]).min()
        print(f"  Distancia mínima parque → Red Natura: {dist_min:.0f} m ({dist_min/1000:.2f} km)")

        # ¿El parque intersecta algún espacio Natura?
        intersecta = natura.geometry.intersects(poligonal.geometry.iloc[0]).any()
        print(f"  El parque intersecta Red Natura: {'SÍ ⚠️' if intersecta else 'NO'}")

        display(natura[[c for c in natura.columns if c != 'geometry']].head())

---
## 7. Asignación de servicios ecosistémicos (CICES v5.1)

Tabla de correspondencia entre clases CORINE Land Cover 2018 y hábitats, con sus
valores unitarios de referencia (€/ha/año) en tres escenarios.

Notas de método:
- Las **superficies artificiales** (111, 112, 121) reciben valor **0** (se excluye de forma conservadora el verde urbano).
- Los valores monetarios son una **síntesis del analista** informada por la literatura de benefit transfer, **no** cifras literales de las fuentes (ver sección 14).
- Cada clase indica su **referencia principal** en la columna `fuente`.


In [ ]:
# ── TABLA DE VALORACIÓN: CORINE Land Cover 2018 × servicios ecosistémicos ─────
#
# Columnas:
#   clc_code   : código CORINE nivel 3
#   clc_label  : descripción CORINE
#   habitat    : tipo de hábitat simplificado
#   val_min, val_central, val_max : €/ha/año (escenarios)
#   fuente     : referencia principal de benefit transfer
#
# IMPORTANTE — los valores €/ha/año NO son cifras literales de las fuentes
# citadas, sino una SÍNTESIS del analista informada por la literatura de
# benefit transfer y adaptada al contexto gallego (juicio técnico; ver
# sección 14, Limitaciones). Las superficies artificiales reciben valor 0.

cices_tabla = pd.DataFrame([
    # CLC   Descripción                          Hábitat                  Min   Cen    Max   Fuente
    ("111", "Continuous urban fabric",           "Superficie artificial",   0,    0,    0,  "—"),
    ("112", "Discontinuous urban fabric",        "Superficie artificial",   0,    0,    0,  "—"),
    ("121", "Industrial / commercial units",     "Superficie artificial",   0,    0,    0,  "—"),
    ("211", "Non-irrigated arable land",         "Cultivos",              120,  210,  350,  "Quintas-Soriano 2016"),
    ("231", "Pastures",                          "Pastizal",              280,  450,  680,  "Quintas-Soriano 2016"),
    ("242", "Complex cultivation patterns",      "Mosaico agrícola",      200,  340,  520,  "de Groot 2012"),
    ("243", "Agriculture w/ natural vegetation", "Mosaico agrícola",      180,  300,  460,  "de Groot 2012"),
    ("311", "Broad-leaved forest",               "Bosque caducifolio",    980, 1650, 2800,  "de Groot 2012 / Costanza 2014"),
    ("312", "Coniferous forest",                 "Bosque coníferas",      550,  920, 1500,  "de Groot 2012"),
    ("313", "Mixed forest",                      "Bosque mixto",          780, 1280, 2100,  "de Groot 2012"),
    ("321", "Natural grasslands",                "Matorral / brezal",     480,  790, 1250,  "Costanza 2014"),
    ("322", "Moors and heathland",               "Matorral / brezal",     480,  790, 1250,  "Costanza 2014"),
    ("323", "Sclerophyllous vegetation",         "Matorral / brezal",     420,  700, 1100,  "Costanza 2014"),
    ("324", "Transitional woodland-shrub",       "Matorral transición",   520,  850, 1350,  "Costanza 2014"),
    ("331", "Beaches, dunes, sands",             "Arenales",               80,  130,  200,  "de Groot 2012"),
    ("333", "Sparsely vegetated areas",          "Escasa vegetación",      50,   80,  130,  "de Groot 2012"),
    ("411", "Inland marshes",                    "Humedal interior",      880, 1480, 2450,  "de Groot 2012"),
    ("412", "Peat bogs",                         "Turbera",               820, 1380, 2280,  "de Groot 2012"),
    ("511", "Water courses",                     "Curso fluvial",         650, 1080, 1750,  "Costanza 2014"),
    ("512", "Water bodies",                      "Masa de agua",          580,  970, 1580,  "Costanza 2014"),
], columns=["clc_code","clc_label_ref","habitat","val_min","val_central","val_max","fuente"])

print(f"Tabla de valoración cargada: {len(cices_tabla)} clases CORINE mapeadas")
display(cices_tabla[["clc_code","habitat","val_min","val_central","val_max","fuente"]])

---
## 8. Valoración monetaria por benefit transfer (línea base — área de influencia)


In [ ]:
if clc is not None:
    # Unir tabla de superficies con tabla de valoración
    clc_sup = tabla_sup.copy()
    clc_sup["clc_code"] = clc_sup[col_codigo].astype(str).str.strip()

    valoracion = clc_sup.merge(cices_tabla, on="clc_code", how="left")

    # Clases que no aparecen en la tabla:
    #   - artificiales (código que empieza por '1') -> valor 0 (conservador)
    #   - resto                                     -> valor de matorral, marcado para revisar
    sin_mapear = valoracion["val_central"].isna()
    if sin_mapear.any():
        print(f"Clases sin entrada en tabla: {valoracion.loc[sin_mapear, 'clc_code'].tolist()}")
        for idx in valoracion.index[sin_mapear]:
            code = str(valoracion.at[idx, "clc_code"])
            if code.startswith("1"):                       # superficie artificial
                valoracion.loc[idx, ["val_min","val_central","val_max"]] = [0, 0, 0]
                valoracion.at[idx, "habitat"] = "Superficie artificial"
                valoracion.at[idx, "fuente"]  = "—"
            else:                                          # natural sin clasificar
                valoracion.loc[idx, ["val_min","val_central","val_max"]] = [480, 790, 1250]
                valoracion.at[idx, "habitat"] = "Sin clasificar (revisar)"
                valoracion.at[idx, "fuente"]  = "supuesto"
        print("  -> artificiales = 0 €/ha/año ; natural sin clasificar = matorral (revisar)")

    # Valor anual total (€/año)
    valoracion["val_anual_min"]     = valoracion["sup_ha"] * valoracion["val_min"]
    valoracion["val_anual_central"] = valoracion["sup_ha"] * valoracion["val_central"]
    valoracion["val_anual_max"]     = valoracion["sup_ha"] * valoracion["val_max"]

    total_anual_min     = valoracion["val_anual_min"].sum()
    total_anual_central = valoracion["val_anual_central"].sum()
    total_anual_max     = valoracion["val_anual_max"].sum()

    print(f"\nValor anual de SE (línea base) — área de influencia 5 km:")
    print(f"  Mínimo:   {total_anual_min:>12,.0f} €/año")
    print(f"  Central:  {total_anual_central:>12,.0f} €/año")
    print(f"  Máximo:   {total_anual_max:>12,.0f} €/año")

---
## 9. VAN a 25 años (tasa de descuento 3%)

El horizonte temporal de 25 años corresponde a la vida útil estándar  
de un parque eólico de esta tipología.

In [ ]:
if clc is not None:
    TASA    = 0.03   # 3% — tasa social de descuento recomendada por CE
    ANOS    = 25

    # Factor de anualidad: suma de (1/(1+r)^t) para t=1..25
    factor_van = sum(1 / (1 + TASA)**t for t in range(1, ANOS + 1))

    VAN_min     = total_anual_min     * factor_van
    VAN_central = total_anual_central * factor_van
    VAN_max     = total_anual_max     * factor_van

    print(f"Factor de anualidad (r={TASA*100:.0f}%, {ANOS} años): {factor_van:.4f}")
    print()
    print(f"VAN de servicios ecosistémicos — área de influencia {RADIO_M/1000:.0f}km:")
    print(f"  VAN mínimo:   {VAN_min:>15,.0f} €")
    print(f"  VAN central:  {VAN_central:>15,.0f} €")
    print(f"  VAN máximo:   {VAN_max:>15,.0f} €")

---
## 9b. Análisis de sensibilidad del VAN

Robustez del VAN central frente a la tasa de descuento y al horizonte temporal.


In [ ]:
if clc is not None:
    filas = []
    for tasa in (0.03, 0.05, 0.07):
        for anos in (25, 30):
            f = sum(1 / (1 + tasa) ** t for t in range(1, anos + 1))
            filas.append({
                "Tasa": f"{tasa*100:.0f}%",
                "Horizonte (años)": anos,
                "Factor anualidad": round(f, 3),
                "VAN central (€)": round(total_anual_central * f),
            })
    sensibilidad = pd.DataFrame(filas)
    print("Sensibilidad del VAN central a tasa de descuento y horizonte:")
    display(sensibilidad.style.format({"VAN central (€)": "{:,.0f}"}))

---
## 10. Valor contenido en la poligonal (proxy de superficie afectada)

Valor de SE de la superficie **dentro del perímetro del parque**, como referencia
territorial adicional.

> **No es el impacto neto.** La huella física real (viales, plataformas, LAT) es
> mucho menor que la poligonal, y los efectos de fragmentación e indirectos se
> tratan en la Pieza 2. Esta cifra es un **límite superior territorial** de la
> poligonal, no la pérdida atribuible al proyecto.


In [ ]:
if clc is not None:
    # Recorte de CORINE a la poligonal del parque
    clc_pol = gpd.clip(clc, poligonal).copy()
    clc_pol["sup_ha"]   = clc_pol.geometry.area / 10_000
    clc_pol["clc_code"] = clc_pol[col_codigo].astype(str).str.strip()

    val_pol = (clc_pol.groupby("clc_code")["sup_ha"].sum().reset_index()
               .merge(cices_tabla[["clc_code","habitat","val_central"]],
                      on="clc_code", how="left"))
    # Artificiales / no mapeadas dentro de la poligonal -> 0 (mismo criterio conservador)
    val_pol["val_central"] = val_pol["val_central"].fillna(0)
    val_pol["val_anual_central"] = val_pol["sup_ha"] * val_pol["val_central"]

    pol_sup           = val_pol["sup_ha"].sum()
    pol_anual_central = val_pol["val_anual_central"].sum()
    pol_van_central   = pol_anual_central * factor_van

    print(f"Superficie de la poligonal analizada: {pol_sup:>10.1f} ha")
    print(f"Valor SE en poligonal (central):      {pol_anual_central:>12,.0f} €/año")
    print(f"VAN SE en poligonal (25a, 3%):        {pol_van_central:>12,.0f} €")
    print("(proxy territorial de la poligonal — NO es el impacto neto del parque)")

---
## 11. Tabla ejecutiva de resultados


In [ ]:
if clc is not None:
    # ══════════════════════════════════════════════════════════════════════════
    # BLOQUE 1 — LÍNEA BASE (área de influencia 5 km)
    # Valor de contexto territorial. NO es el impacto del parque.
    # ══════════════════════════════════════════════════════════════════════════
    base = valoracion[[
        col_codigo, "habitat", "sup_ha", "%_total",
        "val_central", "val_anual_central"
    ]].copy()
    base.columns = [
        "Código CLC", "Hábitat", "Superficie (ha)", "% total",
        "Valor unit. (€/ha/año)", "Valor anual (€/año)"
    ]

    fila_total = pd.DataFrame([[
        "TOTAL", "", round(base["Superficie (ha)"].sum(), 2), 100.0, "",
        base["Valor anual (€/año)"].sum()
    ]], columns=base.columns)

    fila_van = pd.DataFrame([[
        "VAN 25a / 3%", "", np.nan, "", "", round(VAN_central, 0)
    ]], columns=base.columns)

    tabla_base = pd.concat([base, fila_total, fila_van], ignore_index=True)

    # ══════════════════════════════════════════════════════════════════════════
    # BLOQUE 2 — POLIGONAL DEL PARQUE (proxy de superficie afectada)
    # Las 909,6 ha dentro del perímetro. NO es la pérdida atribuible al proyecto.
    # ══════════════════════════════════════════════════════════════════════════
    tabla_poligonal = pd.DataFrame({
        "Concepto": [
            "Superficie de la poligonal",
            "Valor anual de SE (central)",
            "VAN 25a / 3% (central)",
        ],
        "Valor": [
            f"{pol_sup:,.2f} ha",
            f"{pol_anual_central:,.0f} €/año",
            f"{pol_van_central:,.0f} €",
        ],
    })

    # ── Mostrar los dos bloques por separado ──────────────────────────────────
    display(tabla_base.style
        .format({"Valor anual (€/año)": "{:,.0f}", "Superficie (ha)": "{:.2f}"})
        .set_caption("BLOQUE 1 · Línea base (área de influencia 5 km) — "
                     "contexto territorial, NO impacto del parque"))

    print("\n" + "─" * 72)
    print("  Los dos bloques NO se suman: la poligonal está DENTRO del área de")
    print("  influencia. Son dos lecturas del mismo territorio a distinta escala.")
    print("─" * 72 + "\n")

    display(tabla_poligonal.style
        .hide(axis="index")
        .set_caption("BLOQUE 2 · Poligonal del parque (proxy) — "
                     "NO es la pérdida atribuible al proyecto (eso es la Pieza 2)"))

---
## 12. Exportación a Excel


In [ ]:
if clc is not None:
    EXCEL_OUT = "paxareiras_ii_valoracion_SE.xlsx"

    resumen = pd.DataFrame({
        "Parámetro": [
            "Expediente", "Promotor", "Municipio",
            "Potencia instalada (MW)", "Nº aerogeneradores",
            "Red soterrada (m)", "Línea evacuación (m)",
            "Área influencia (ha)", "Metodología",
            "Tasa descuento", "Horizonte temporal (años)",
            "Valor anual mín — línea base (€/año)",
            "Valor anual central — línea base (€/año)",
            "Valor anual máx — línea base (€/año)",
            "VAN mínimo — línea base (€)",
            "VAN central — línea base (€)",
            "VAN máximo — línea base (€)",
            "Valor anual central — poligonal proxy (€/año)",
            "VAN central — poligonal proxy (€)",
        ],
        "Valor": [
            "IN661A 2011/8 (NT)", "AV Paxareiras, S.L.", "Dumbría, A Coruña",
            26.40, 5,
            5967, 7255,
            round(sup_influencia, 1), "CICES v5.1 + Benefit Transfer",
            "3%", 25,
            round(total_anual_min), round(total_anual_central), round(total_anual_max),
            round(VAN_min), round(VAN_central), round(VAN_max),
            round(pol_anual_central), round(pol_van_central),
        ]
    })

    with pd.ExcelWriter(EXCEL_OUT, engine="openpyxl") as writer:
        resumen.to_excel(writer, sheet_name="Resumen", index=False)
        tabla_base.to_excel(writer, sheet_name="Linea base", index=False)
        tabla_poligonal.to_excel(writer, sheet_name="Poligonal proxy", index=False)
        sensibilidad.to_excel(writer, sheet_name="Sensibilidad VAN", index=False)
        valoracion.drop(columns="geometry", errors="ignore").to_excel(
            writer, sheet_name="Detalle por clase CLC", index=False
        )

    print(f"✅ Exportado: {EXCEL_OUT}")

---
## 13. Visualizaciones de publicación


In [ ]:
if clc is not None:
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # ── MAPA IZQUIERDA: usos del suelo + elementos del parque ──
    ax1 = axes[0]
    clc.plot(column=col_codigo, ax=ax1, alpha=0.65,
             edgecolor="white", linewidth=0.2, cmap="tab20",
             legend=True,
             legend_kwds={"title":"Código CLC","loc":"lower right","fontsize":7})
    area_influencia.boundary.plot(ax=ax1, color="red", linewidth=1.8,
                                  linestyle="--", label="Área influencia 5km")
    poligonal.boundary.plot(ax=ax1, color="black", linewidth=2, label="Poligonal parque")
    aeroxeradores.plot(ax=ax1, color="yellow", edgecolor="black",
                       markersize=70, marker="^", zorder=5, label="Aerogeneradores")
    linea_evac.plot(ax=ax1, color="orange", linewidth=1.5, label="Línea evacuación")

    if CONTEXTILY:
        ctx.add_basemap(ax1, crs=CRS, source=ctx.providers.CartoDB.Positron, zoom=12)

    ax1.set_title("Usos del suelo CORINE 2018\nPaxareiras II — IN661A 2011/8",
                  fontsize=11, fontweight="bold")
    ax1.legend(loc="upper left", fontsize=8)
    ax1.set_xlabel("Este (m) UTM 29N")
    ax1.set_ylabel("Norte (m) UTM 29N")

    # ── GRÁFICO DERECHA: barras de valor por hábitat ──
    ax2 = axes[1]
    plot_data = (valoracion.groupby("habitat")[["val_anual_min","val_anual_central","val_anual_max"]]
                 .sum().sort_values("val_anual_central", ascending=True))

    y = np.arange(len(plot_data))
    bar_w = 0.25
    ax2.barh(y - bar_w, plot_data["val_anual_min"]/1000,    bar_w, label="Mínimo",  color="#a8d5ba")
    ax2.barh(y,          plot_data["val_anual_central"]/1000, bar_w, label="Central", color="#2d936c")
    ax2.barh(y + bar_w, plot_data["val_anual_max"]/1000,    bar_w, label="Máximo",  color="#1a5c3a")

    ax2.set_yticks(y)
    ax2.set_yticklabels(plot_data.index, fontsize=9)
    ax2.set_xlabel("Valor anual (miles €/año)")
    ax2.set_title("Valor anual de SE por hábitat — línea base\n(escenarios mín/central/máx)",
                  fontsize=11, fontweight="bold")
    ax2.legend(fontsize=9)
    ax2.grid(axis="x", alpha=0.3)

    plt.tight_layout()
    plt.savefig("paxareiras_ii_valoracion_SE.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Figura guardada: paxareiras_ii_valoracion_SE.png")

---
## 14. Limitaciones metodológicas y referencias

> Este apartado sustenta por qué el resultado es una **aproximación de primer orden** (ver *Finalidad y alcance*). Declarar estas limitaciones forma parte del rigor del trabajo, no lo debilita.

**Sobre el marco y la decisión:**

1. **Vacío normativo.** El Art. 10.3 de la Ley 2/2024 habilita directrices metodológicas para la valoración de servicios ecosistémicos, pero la Xunta de Galicia **aún no las ha publicado**. En ausencia de tablas oficiales, esta valoración descansa en **juicio técnico del analista**.
2. **Línea base, no impacto.** Se valora el *stock* de servicios del territorio, no el *cambio marginal* atribuible al parque. Las decisiones de EIA dependen del valor marginal; el lado del impacto (huella, fragmentación, colisión de avifauna/quirópteros, paisaje, balance de carbono) queda fuera de esta pieza.

**Sobre la transferencia de valores (benefit transfer):**

3. **Método básico (unit value transfer).** Se transfiere un valor único por clase; es el escalón de menor precisión, con errores documentados del orden del 20–100%.
4. **Encaje de sitio (site mismatch).** Las fuentes son agregados globales (Costanza et al. 2014; de Groot et al. 2012) o de zonas áridas del SE peninsular (Quintas-Soriano et al. 2016), ecológicamente distintas de la Galicia atlántica húmeda. Lo idóneo sería literatura primaria atlántica/templada.
5. **Valores = síntesis del analista.** Las cifras €/ha/año son una síntesis informada por la literatura, **no** extracciones literales de las fuentes.
6. **CICES nominal.** El marco se cita, pero la valoración es por hábitat (valor *bundled*), no por servicio finalista CICES; esto no controla el riesgo de doble conteo ni permite ligar servicios concretos a impactos concretos.

**Sobre los datos espaciales:**

7. **Resolución de CORINE.** Unidad mínima de 25 ha y precisión de 100 m: demasiado grueso para una poligonal de ~910 ha. Cartografías superiores para Galicia: SIOSE/IGVG y, sobre todo, los Hábitats de Interese Comunitario (Anexo I, Dir. 92/43/CEE) del Plan Director da Rede Natura.
8. **Plantación ≠ bosque natural.** En Galicia, las clases CORINE 312/313 son mayoritariamente plantaciones de pino y eucalipto; aplicarles valores de bosque templado **sobrevalora** sus servicios. Solo el 311 (frondosa autóctona) justifica el valor alto.

**Sobre el descuento:**

9. **Tasa y horizonte.** Se aplica el horizonte financiero del parque (25 a / 3%) a un stock de capital natural. El debate en economía ambiental favorece tasas **bajas y decrecientes** para capital natural; no se trata la irreversibilidad ni el valor de opción.

**Refinamientos pendientes:** reindexación por IPC; propagación de incertidumbre tipo Monte Carlo sobre los valores unitarios (en lugar de un rango mín/central/máx fijado por el analista).

**Referencias:**

- Costanza, R., de Groot, R., Sutton, P., et al. (2014). *Changes in the global value of ecosystem services.* Global Environmental Change, 26, 152–158.
- de Groot, R., Brander, L., van der Ploeg, S., et al. (2012). *Global estimates of the value of ecosystems and their services in monetary units.* Ecosystem Services, 1(1), 50–61.
- Quintas-Soriano, C., Castro, A.J., Castro, H., García-Llorente, M. (2016). *Impacts of land use change on ecosystem services and implications for human well-being in Spanish drylands.* Land Use Policy, 54, 534–548.
- Haines-Young, R. & Potschin, M. (2018). *CICES v5.1: Guidance on the application of the revised structure.*


---
## Estado del proyecto

✅ **Pieza 1 — Sesión 1:** Estructura base + coordenadas oficiales + carga de datos  
✅ **Pieza 1 — Sesión 2:** Valoración CICES + VAN + tabla ejecutiva + Excel + mapa  
✅ **Pieza 1 — Revisión:** superficies artificiales a 0, fuentes por clase, sensibilidad del VAN, proxy poligonal, limitaciones y vacío normativo  

### Pendiente para publicación GitHub:
- [ ] **Re-ejecutar el notebook completo con los shapefiles reales** (la lógica ha cambiado: las salidas guardadas se han limpiado)
- [ ] Decidir el encuadre final de la sección 10 (proxy poligonal) y, si procede, refinar hacia la huella física real
- [ ] Reindexar valores por IPC (opcional, declarado como limitación)
- [ ] `README.md` + `requirements.txt` del repositorio
- [ ] Pieza 2 — Fragmentación de hábitats

**Deadline: finales de julio 2026**
